In [ ]:
"""
NOTEBOOK: PRICE PREDICTION MODEL
=================================
Purpose: Predict crop prices using time series models
Output: ARIMA and Deep Learning models
"""

# 💰 Price Prediction Model Notebook

**Objective:** Forecast crop prices for next 7-30 days
**Models:** ARIMA (baseline), LSTM (advanced), Prophet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

class PricePredictor:
    """
    PRODUCTION-READY PRICE PREDICTION MODEL
    Exports to: backend/app/ml/price_predictor.py
    """
    
    def __init__(self):
        self.models = {}
        self.results = {}
    
    def prepare_time_series(self, df, crop_type, location):
        """Prepare time series data for a specific crop and location"""
        crop_data = df[(df['crop_type'] == crop_type) & (df['location'] == location)]
        crop_data = crop_data.sort_values('date')
        
        # Aggregate by day
        daily_prices = crop_data.groupby('date')['price_per_kg'].mean()
        
        return daily_prices
    
    def train_arima(self, series, crop_type, location):
        """Train ARIMA model for price prediction"""
        print(f"📈 Training ARIMA for {crop_type} in {location}...")
        
        # Fit ARIMA model (p=5, d=1, q=0 for MVP)
        model = ARIMA(series, order=(5, 1, 0))
        model_fit = model.fit()
        
        # Make predictions
        predictions = model_fit.forecast(steps=30)
        
        # Calculate metrics on training data
        fitted_values = model_fit.fittedvalues
        mae = mean_absolute_error(series[len(series)-len(fitted_values):], fitted_values)
        
        self.models[f'arima_{crop_type}_{location}'] = model_fit
        
        return {
            'model': model_fit,
            'predictions': predictions,
            'mae': mae,
            'aic': model_fit.aic
        }
    
    def train_prophet(self, df, crop_type, location):
        """Train Facebook Prophet model"""
        print(f"📊 Training Prophet for {crop_type} in {location}...")
        
        # Prepare data for Prophet
        crop_data = df[(df['crop_type'] == crop_type) & (df['location'] == location)]
        prophet_df = pd.DataFrame({
            'ds': pd.to_datetime(crop_data['date']),
            'y': crop_data['price_per_kg']
        })
        
        # Train model
        model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
        model.fit(prophet_df)
        
        # Make future dataframe
        future = model.make_future_dataframe(periods=30)
        forecast = model.predict(future)
        
        self.models[f'prophet_{crop_type}_{location}'] = model
        
        return {
            'model': model,
            'forecast': forecast,
            'model_name': 'prophet'
        }
    
    def predict_price(self, crop_type, location, days_ahead=7):
        """Predict price for given crop and location"""
        model_key = f'arima_{crop_type}_{location}'
        
        if model_key in self.models:
            model = self.models[model_key]
            predictions = model.forecast(steps=days_ahead)
            
            return {
                'crop': crop_type,
                'location': location,
                'predictions': predictions.tolist(),
                'confidence_lower': (predictions * 0.9).tolist(),
                'confidence_upper': (predictions * 1.1).tolist()
            }
        
        return None
    
    def evaluate_model(self, actual, predicted):
        """Evaluate prediction accuracy"""
        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mape = np.mean(np.abs((actual - predicted) / actual)) * 100
        
        return {
            'mae': mae,
            'rmse': rmse,
            'mape': mape,
            'accuracy': 100 - mape
        }
    
    def save_model(self, version="v1"):
        """Save trained models to production"""
        import joblib
        import os
        
        # NOTE: Made paths relative to where they ran previously so Jupyter would save properly
        os.makedirs("../../backend/ml_weights", exist_ok=True)
        
        for model_name, model in self.models.items():
            path = f"../../backend/ml_weights/price_{model_name}_{version}.pkl"
            joblib.dump(model, path)
            print(f"💾 Saved: {path}")
        
        print(f"✅ All price models saved (version {version})")

print("\n✅ Price prediction models ready!")